<a href="https://colab.research.google.com/github/heyanugrah/CNNronaldoMessiClassification/blob/main/mytransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Basic Transformer- Encoder

In [18]:
import torch
import torch.nn as nn
import math

In [19]:
# 1. Define the Vocabulary (Adding '[sep]')
new_vocab = {'<pad>': 0, '<unk>': 1, '<cls>': 2, '[sep]': 3, 'this': 4, 'is': 5, 'a': 6, 'positive': 7, 'sentence': 8, 'learning': 9, 'about': 10, 'transformers': 11, 'fun': 12, 'i': 13, 'hate': 14, 'am': 15, 'so': 16, 'disappointed': 17, 'really': 18, 'enjoyed': 19, 'movie': 20, 'wonderful': 21, 'day': 22, 'for': 23, 'walk': 24, 'product': 25, 'excellent!': 26, 'absolutely': 27, 'dreadful': 28, 'service': 29, 'what': 30, 'terrible': 31, 'experience': 32, 'feeling': 33, 'very': 34, 'happy': 35, 'today': 36, 'boy': 37, 'anugrah': 38}

# 2. Define the Reverse Vocabulary
idx_to_word = {i: word for word, i in new_vocab.items()}

vocab_size = len(new_vocab)

print(f"Vocabulary size is: {vocab_size}")

Vocabulary size is: 39


### Modular Tokenization Functions

To improve clarity and reusability, let's break down the tokenization process into several distinct functions:

1.  `_add_special_tokens_and_segments`: Adds `<cls>` and `[sep]` tokens and initializes segment IDs.
2.  `_convert_to_ids`: Converts token strings to their numerical IDs.
3.  `_pad_ids_and_segments`: Handles padding for both token IDs and segment IDs.
4.  `_create_attention_mask_from_ids`: Generates the attention mask from padded token IDs.
5.  `prepare_transformer_inputs`: An orchestrating function that calls the above helpers to produce all necessary inputs for a Transformer model.

In [20]:
# Helper to add special tokens and initial segment IDs
def _add_special_tokens_and_segments(input_text, vocab):
    tokens = [vocab.get("<cls>", "<cls>")] # Start with <cls>
    segment_ids = [0] # <cls> belongs to segment 0

    # Tokenize input text and add words
    for word in input_text.lower().replace('.', '').split():
        tokens.append(word)
        segment_ids.append(0) # All words in the first sentence belong to segment 0

    tokens.append(vocab.get("[sep]", "[sep]")) # Add [sep]
    segment_ids.append(0) # [sep] belongs to segment 0

    return tokens, segment_ids

# Helper to convert tokens to IDs
def _convert_to_ids(tokens, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

# Helper to pad token IDs and segment IDs
def _pad_ids_and_segments(token_ids, segment_ids, max_seq_len, pad_token_id):
    if len(token_ids) > max_seq_len:
        token_ids = token_ids[:max_seq_len]
        segment_ids = segment_ids[:max_seq_len]
    else:
        padding_length = max_seq_len - len(token_ids)
        token_ids.extend([pad_token_id] * padding_length)
        segment_ids.extend([0] * padding_length) # Padding tokens belong to segment 0

    return token_ids, segment_ids

# Helper to create attention mask
def _create_attention_mask_from_ids(token_ids, pad_token_id):
    mask = [1 if token_id != pad_token_id else 0 for token_id in token_ids]
    return torch.tensor(mask, dtype=torch.long).unsqueeze(0).unsqueeze(1).unsqueeze(2) # [batch_size, 1, 1, seq_len]

In [21]:
def prepare_transformer_inputs(input_text, vocab, max_seq_len):
    pad_token_id = vocab['<pad>']

    # Step 1: Add special tokens and assign initial segment IDs
    tokens, segment_ids = _add_special_tokens_and_segments(input_text, vocab)

    # Step 2: Convert tokens to IDs
    token_ids = _convert_to_ids(tokens, vocab)

    # Step 3: Pad token IDs and segment IDs
    padded_token_ids, padded_segment_ids = _pad_ids_and_segments(
        token_ids, segment_ids, max_seq_len, pad_token_id
    )

    # Convert to PyTorch tensors and add batch dimension
    input_ids = torch.tensor(padded_token_ids, dtype=torch.long).unsqueeze(0)
    token_type_ids = torch.tensor(padded_segment_ids, dtype=torch.long).unsqueeze(0)

    # Step 4: Create attention mask
    attention_mask = _create_attention_mask_from_ids(padded_token_ids, pad_token_id)

    return input_ids, attention_mask, token_type_ids

### Demonstration of the new tokenization pipeline

In [22]:


input_text = "anugrah is learning"
max_seq_len = 10 # maximum token need to be created

# Prepare inputs using the new modular function
input_ids, attention_mask_new, token_type_ids_new = prepare_transformer_inputs(
    input_text, new_vocab, max_seq_len
)

print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids}")
print(f"Attention Mask: {attention_mask_new}")
print(f"Token Type IDs: {token_type_ids_new}")

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Mask shape: {attention_mask_new.shape}")
print(f"Token Type IDs shape: {token_type_ids_new.shape}")

Input Text: 'anugrah is learning'
Input IDs: tensor([[ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0]])
Attention Mask: tensor([[[[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]]])
Token Type IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Input IDs shape: torch.Size([1, 10])
Attention Mask shape: torch.Size([1, 1, 1, 10])
Token Type IDs shape: torch.Size([1, 10])


### Create and apply embedding

In [23]:
d_model = 8 #vector dimension for each word

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

embedded_input = embedding(input_ids)
print(f"Shape of embedded input: {embedded_input.shape}")

Shape of embedded input: torch.Size([1, 10, 8])


In [24]:
print(embedded_input)

tensor([[[ 0.7887,  1.2992,  0.1793, -0.3169,  0.5662,  0.5977,  1.7248,
          -0.7146],
         [ 0.8307, -0.7291,  0.7234,  0.6115,  0.1040,  0.4022,  0.6057,
          -0.0052],
         [-0.9653,  0.2254,  1.4016, -0.1021,  0.8869,  0.7399,  0.2378,
           0.8444],
         [ 0.3838, -0.3940,  0.0389, -0.9575, -1.3712,  0.1241,  0.2155,
          -0.0454],
         [ 0.7887,  1.2992,  0.1793, -0.3169,  0.5662,  0.5977,  1.7248,
          -0.7146],
         [-1.5892,  2.0352, -0.3876,  1.2208,  0.3285, -0.7065, -0.7202,
          -0.1505],
         [-1.5892,  2.0352, -0.3876,  1.2208,  0.3285, -0.7065, -0.7202,
          -0.1505],
         [-1.5892,  2.0352, -0.3876,  1.2208,  0.3285, -0.7065, -0.7202,
          -0.1505],
         [-1.5892,  2.0352, -0.3876,  1.2208,  0.3285, -0.7065, -0.7202,
          -0.1505],
         [-1.5892,  2.0352, -0.3876,  1.2208,  0.3285, -0.7065, -0.7202,
          -0.1505]]], grad_fn=<EmbeddingBackward0>)


In [25]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nEncodings for each token in 'anugrah is learning':")
# Iterate through the input_ids and corresponding embedded vectors
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    embedding_vector = embedded_input.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Embedding: {embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Encodings for each token in 'anugrah is learning':
Token: '<unk>' (ID: 1) -> Embedding: [0.7887090444564819, 1.2991664409637451, 0.1792662888765335, -0.3168502449989319, 0.5662166476249695, 0.5977156758308411, 1.7247928380966187, -0.7146061658859253]
Token: 'anugrah' (ID: 38) -> Embedding: [0.8307034373283386, -0.7290827035903931, 0.723368763923645, 0.6114864945411682, 0.104031041264534, 0.4022086560726166, 0.6056692600250244, -0.005194216966629028]
Token: 'is' (ID: 5) -> Embedding: [-0.9652851819992065, 0.22536695003509521, 1.4015508890151978, -0.10205242037773132, 0.8869385123252869, 0.7398892641067505, 0.23782329261302948, 0.8444193601608276]
Token: 'learning' (ID: 9) -> Embedding: [0.3837636411190033, -0.39399614930152893, 0.03894075006246567, -0.9575411677360535, -1.371206283569336, 0.12409716844558716, 0.2154805213212967, -0.04539754241704941]
Token: '<unk>' (ID: 1) -> Embedding: [0.788

In [26]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super().__init__()

        self.dropout = nn.Dropout(0.1)

        # Create a positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term) if d_model % 2 == 0 else torch.cos(position * div_term[:-1])

        pe = pe.unsqueeze(0) # Add batch dimension
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add positional encoding to the input embeddings
        # x has shape (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Instantiate Positional Encoding
pos_encoder = PositionalEncoding(d_model, max_seq_len)

# Apply positional encoding to the embedded input
embedded_with_pos = pos_encoder(embedded_input)

print(f"Shape of embedded input with positional encoding: {embedded_with_pos.shape}")

Shape of embedded input with positional encoding: torch.Size([1, 10, 8])


In [27]:
print(embedded_with_pos[0][1],embedded_with_pos.shape) #PE for anugrah (1,batch,10 total , 8 dimension)

tensor([ 1.8580, -0.2098,  0.9147,  1.7850,  0.1267,  0.0000,  0.6741,  1.1053],
       grad_fn=<SelectBackward0>) torch.Size([1, 10, 8])


In [28]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nCombined Embeddings (Word Embedding + Positional Encoding) for each token:")
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    combined_embedding_vector = embedded_with_pos.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Combined Embedding: {combined_embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Combined Embeddings (Word Embedding + Positional Encoding) for each token:
Token: '<unk>' (ID: 1) -> Combined Embedding: [0.8763434290885925, 2.5546295642852783, 0.19918477535247803, 0.7590553164482117, 0.6291296482086182, 1.7752397060394287, 1.9164365530014038, 0.3171042799949646]
Token: 'anugrah' (ID: 38) -> Combined Embedding: [1.8579716682434082, -0.20975597202777863, 0.9146691560745239, 1.784989595413208, 0.1267009824514389, 0.0, 0.6740769743919373, 1.1053392887115479]
Token: 'is' (ID: 5) -> Combined Embedding: [-0.062208641320466995, -0.21197767555713654, 1.7780225276947021, 0.9755713939666748, 0.0, 0.0, 0.266470342874527, 2.0493526458740234]
Token: 'learning' (ID: 9) -> Combined Embedding: [0.5832040309906006, -1.5377652645111084, 0.37162327766418457, -0.0, -1.490234375, 1.2484970092773438, 0.24275614321231842, 1.0606645345687866]
Token: '<unk>' (ID: 1) -> Combined Embedding: [0.035451

In [29]:
token_type_embedding = nn.Embedding(2, d_model) # Assuming 2 types: 0 for sentence A, 1 for sentence B

'''
Segment Embedding

Tells the model which sentence a token belongs to.

Example:

Sentence A: I love AI
Sentence B: It is powerful

BERT input:

[CLS] I love AI [SEP] It is powerful [SEP]
'''

# Get segment embeddings for our token_type_ids
segment_embeddings = token_type_embedding(token_type_ids_new)

# Add the segment embeddings to the combined word and positional embeddings
final_transformer_input = embedded_with_pos + segment_embeddings

print(f"Shape of token_type_ids: {token_type_ids_new.shape}")
print(f"Shape of segment embeddings: {segment_embeddings.shape}")
print(f"Shape of final input to Transformer: {final_transformer_input.shape}")

print("\nSegment Embeddings:")
print(segment_embeddings)

Shape of token_type_ids: torch.Size([1, 10])
Shape of segment embeddings: torch.Size([1, 10, 8])
Shape of final input to Transformer: torch.Size([1, 10, 8])

Segment Embeddings:
tensor([[[-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
          -0.7631],
         [-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
          -0.7631],
         [-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
          -0.7631],
         [-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
          -0.7631],
         [-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
          -0.7631],
         [-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
          -0.7631],
         [-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
          -0.7631],
         [-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
          -0.7631],
         [-0.3854,  0.6762,  0.0388, -0.9538,  1.1850, -0.8190, -0.1942,
     

In [30]:
print('final transformer input')
print(final_transformer_input)

final transformer input
tensor([[[ 0.4910,  3.2308,  0.2380, -0.1947,  1.8142,  0.9563,  1.7222,
          -0.4460],
         [ 1.4726,  0.4664,  0.9535,  0.8312,  1.3117, -0.8190,  0.4799,
           0.3422],
         [-0.4476,  0.4642,  1.8168,  0.0218,  1.1850, -0.8190,  0.0723,
           1.2862],
         [ 0.1978, -0.8616,  0.4105, -0.9538, -0.3052,  0.4295,  0.0486,
           0.2975],
         [-0.3499,  1.3934,  0.6707, -0.2824,  1.8586,  0.9554,  1.7267,
          -0.7631],
         [-3.2166,  3.2527,  0.1409,  1.3778,  1.6056, -0.8190, -0.9888,
           0.1808],
         [-2.4616,  0.6762,  0.2356,  1.3197,  1.6166, -0.4948, -0.9877,
           0.1808],
         [-1.4212,  3.7752,  0.3240,  1.2525,  1.6277, -0.4955, -0.9866,
          -0.7631],
         [-1.0519,  2.7759,  0.4052,  1.1768,  1.1850, -0.4964, -0.9855,
           0.1808],
         [-1.6933,  1.9252,  0.4785,  1.0933,  1.6499, -0.4973, -0.1942,
           0.1808]]], grad_fn=<AddBackward0>)


**Practise Fields**


In [31]:

Wq = nn.Linear(8, 8)
print(f"Wq (the linear layer module): {Wq}")

# Create a dummy input tensor. The last dimension must match Wq's input features (8).
# Let's use a batch size of 1 and a sequence length of 5.
dummy_input = torch.randn(1, 5, 8)
print(f"\nShape of dummy_input: {dummy_input.shape}")

# Pass the dummy_input through Wq to get an output tensor
Q_output_tensor = Wq(dummy_input)
print(f"Shape of Q_output_tensor (output from Wq): {Q_output_tensor.shape}")

# Now Q_output_tensor is a tensor, and you can call .view() on it
# For example, let's reshape it to (1, 5, 2, 4) if it makes sense for your model
# (1, batch_size; 5, seq_len; 2, num_heads; 4, head_dim)
reshaped_Q = Q_output_tensor.view(1, 5, 2, 4)
print(f"Shape after reshaping with .view(): {reshaped_Q.shape}")

Wq (the linear layer module): Linear(in_features=8, out_features=8, bias=True)

Shape of dummy_input: torch.Size([1, 5, 8])
Shape of Q_output_tensor (output from Wq): torch.Size([1, 5, 8])
Shape after reshaping with .view(): torch.Size([1, 5, 2, 4])


In [32]:
Wq.weight.shape , Wq.bias.shape

(torch.Size([8, 8]), torch.Size([8]))

In [35]:
import torch
import torch.nn as nn
import math

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()

        '''
          d_model = 512
          num_heads = 8

          head_dim = 64
          512 dimensions
          ├─ Head 1 : 64
          ├─ Head 2 : 64
          ├─ Head 3 : 64
          ├─ Head 4 : 64
          ├─ Head 5 : 64
          ├─ Head 6 : 64
          ├─ Head 7 : 64
          └─ Head 8 : 64
        '''

        assert d_model % num_heads == 0 #if it is divisible then return true and allow run

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Multi-Head Attention
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after MHA residual
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.ffn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after FFN residual
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim) # The division by sqrt(head_dim) is the scaling factor.

        if mask is not None:
            # Apply mask to the scores. Masked positions should be set to -inf.
            # This code is used to hide certain tokens from attention.1allowed 0 not allowed
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output

    def forward(self, x, mask=None):
        # In your current setup (cell cbe1d5c1), `mask` is indeed passed as a tensor
        # (attention_mask_new), not None, when calling this layer.

        batch_size, seq_len, _ = x.shape

        # ----------------------------
        # Multi-Head Self Attention
        # ----------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        '''
        Full token embedding (8 dims)
                  │
                  ▼
            Split into 2 heads
                  │
          ┌──────┴──────┐
          ▼             ▼
        Head 1        Head 2
        (4 dims)      (4 dims)
          │             │
        Attention    Attention
        separately   separately
        '''

        '''
        # Split Q into multiple attention heads and rearrange dimensions
        # From: [batch_size, seq_len, d_model]
        # To:   [batch_size, num_heads, seq_len, head_dim]
        '''
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_output = self._calculate_attention(
            Q,
            K,
            V,
            mask=mask
        )
        '''
        Before attention:

        Token
        [1,2,3,4,5,6,7,8]

        ↓ split

        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        After attention:
        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        ↓ concatenate
        combined attention heads : [1,2,3,4,5,6,7,8]
        '''
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model
            )
        )
        '''
        Multi-head outputs
              ↓
        Concatenate
              ↓
        Wo (Linear Layer)
              ↓
        Final attention representation

        [1,2,3,4,5,6,7,8]
            ↓
        Linear Wo
            ↓
       [5,-2,8,1,4,9,3,7]
        '''

        mha_output = self.Wo(
            attention_output
        )
        '''
        Some output values are randomly set to 0.Prevent Overfitting

        Before:
        [5,-2,8,1,4,9,3,7]

        After dropout:
        [5,0,8,1,0,9,3,7]
        '''
        final_mha_output = self.attn_dropout(
            mha_output
        )

        '''
          Input
            ↓
          Wq, Wk, Wv
            ↓
          Split into heads
            ↓
          Attention
            ↓
          Merge heads
            ↓
          Wo
            ↓
          Dropout
            ↓
          Final MHA output
        '''

        # Residual Connection + LayerNorm
        x = self.layer_norm1(
            x + final_mha_output
        )

        # ----------------------------
        # Feed Forward Network
        # ----------------------------

        ffn_output = self.ffn(x)

        ffn_output = self.ffn_dropout(
            ffn_output
        )

        # Residual Connection + LayerNorm
        output = self.layer_norm2(
            x + ffn_output
        )

        return output

# --- Execution Code ---
encoder_layer = TransformerEncoderLayer(d_model=8, num_heads=2, ffn_hidden_dim=32)
encoder_output = encoder_layer(final_transformer_input, mask=attention_mask_new)

print(f"Encoder Layer Input Shape: {final_transformer_input.shape}")
print(f"Encoder Layer Output Shape: {encoder_output.shape}")
print("\nFirst few values of the output:")
print(encoder_output[0, 0, :])

Encoder Layer Input Shape: torch.Size([1, 10, 8])
Encoder Layer Output Shape: torch.Size([1, 10, 8])

First few values of the output:
tensor([-0.8099,  2.1748, -0.0452, -1.3680,  0.5271, -0.3351,  0.3406, -0.4844],
       grad_fn=<SelectBackward0>)
